
# Задачи t7–t10

- **t7** - критерий Пирсона для гипотезы о распределении Пуассона с параметром, найденным через **ОМПГ** по полной группе событий  
  \[
  \{0\},\{1\},\{2\},\{3\},\{4,5, ...}
  \]
- **t8** - проверка **гипотезы независимости** 
- **t9** - проверка **гипотезы однородности**
- **t10a** - проверка согласия с **равномерным** законом
- **t10b** - проверка согласия с **нормальным** законом:
  - критерий Пирсона с параметрами, найденными только через **ОМПГ**
  - критерий Колмогорова для сложной гипотезы через **параметрический bootstrap**


In [1]:

import math
import numpy as np
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import chi2, norm, kstwobign

np.set_printoptions(precision=6, suppress=True)


## t7. Гипотеза о распределении Пуассона, параметр через ОМПГ

In [2]:
# Полная группа событий:
# A0 = {0}, A1 = {1}, A2 = {2}, A3 = {3}, A4 = {k : k >= 4} = {4,5,...}

m7 = np.array([109, 65, 22, 3, 1], dtype=float)
n7 = m7.sum()


def poisson_group_probs(lam):
    p0 = math.exp(-lam)
    p1 = lam * p0
    p2 = lam**2 / 2 * p0
    p3 = lam**3 / 6 * p0
    p4 = 1 - (p0 + p1 + p2 + p3)   # p4 = P(X >= 4)
    return np.array([p0, p1, p2, p3, p4], dtype=float)


def neg_loglik_t7(lam):
    if lam <= 0:
        return np.inf
    p = poisson_group_probs(lam)
    if np.any(p <= 0):
        return np.inf
    return -(m7 * np.log(p)).sum()


res7 = minimize_scalar(neg_loglik_t7, bounds=(1e-9, 10), method="bounded")
lam_hat_7 = res7.x
p7 = poisson_group_probs(lam_hat_7)
exp7 = n7 * p7

# После проверки ожидаемых частот объединяем группы:
# B0 = {0}, B1 = {1}, B2 = {k : k >= 2} = {2,3,4,...}
obs7_merged = np.array([m7[0], m7[1], m7[2] + m7[3] + m7[4]], dtype=float)
exp7_merged = np.array(
    [exp7[0], exp7[1], exp7[2] + exp7[3] + exp7[4]], dtype=float)

delta7 = ((obs7_merged - exp7_merged) ** 2 / exp7_merged).sum()
df7 = 3 - 1 - 1
pval7 = 1 - chi2.cdf(delta7, df=df7)

print("t7")
print("n =", n7)
print("lambda_hat (ОМПГ) =", lam_hat_7)
print("probabilities on full group =", p7)
print("expected on full group =", exp7)
print("merged observed =", obs7_merged)
print("merged expected =", exp7_merged)
print("delta =", delta7)
print("df =", df7)
print("p-value =", pval7)

t7
n = 200.0
lambda_hat (ОМПГ) = 0.6106633666991588
probabilities on full group = [0.542991 0.331584 0.101243 0.020609 0.003573]
expected on full group = [108.59811   66.316887  20.248647   4.121702   0.714654]
merged observed = [109.  65.  26.]
merged expected = [108.59811   66.316887  25.085003]
delta = 0.0610126417701185
df = 1
p-value = 0.8049025188744007


## t8. Проверка гипотезы независимости

In [3]:

# t8
obs8 = np.array([
    [25, 50, 25],
    [52, 41, 7]
], dtype=float)

row8 = obs8.sum(axis=1)
col8 = obs8.sum(axis=0)
n8 = obs8.sum()

exp8 = np.outer(row8, col8) / n8
delta8 = ((obs8 - exp8) ** 2 / exp8).sum()
df8 = (obs8.shape[0] - 1) * (obs8.shape[1] - 1)
pval8 = 1 - chi2.cdf(delta8, df=df8)

print("t8")
print("n =", n8)
print("row sums =", row8)
print("col sums =", col8)
print("expected =")
print(exp8)
print("delta =", delta8)
print("df =", df8)
print("p-value =", pval8)


t8
n = 200.0
row sums = [100. 100.]
col sums = [77. 91. 32.]
expected =
[[38.5 45.5 16. ]
 [38.5 45.5 16. ]]
delta = 20.48264235764236
df = 2
p-value = 3.5665697735942636e-05


## t9. Проверка гипотезы однородности

In [4]:
# t9
# Проверка гипотезы однородности.
# При H0 обе выборки имеют одно и то же распределение, поэтому вероятности p_j оцениваются по объединённой выборке.

obs9 = np.array([
    [33, 43, 80, 144],
    [39, 35, 72, 154]
], dtype=float)

row9 = obs9.sum(axis=1)
col9 = obs9.sum(axis=0)
n9 = obs9.sum()

p9 = col9 / n9
exp9 = np.outer(row9, p9)

delta1 = ((obs9[0] - exp9[0]) ** 2 / exp9[0]).sum()
delta2 = ((obs9[1] - exp9[1]) ** 2 / exp9[1]).sum()
delta9 = delta1 + delta2

df9 = (obs9.shape[0] - 1) * (obs9.shape[1] - 1)
pval9 = 1 - chi2.cdf(delta9, df=df9)

print("t9")
print("n =", n9)
print("estimated common probabilities =", p9)
print("expected =")
print(exp9)
print("delta_1 =", delta1)
print("delta_2 =", delta2)
print("delta =", delta9)
print("df =", df9)
print("p-value =", pval9)

t9
n = 600.0
estimated common probabilities = [0.12     0.13     0.253333 0.496667]
expected =
[[ 36.  39.  76. 149.]
 [ 36.  39.  76. 149.]]
delta_1 = 1.0385679609452128
delta_2 = 1.0385679609452128
delta = 2.0771359218904255
df = 3
p-value = 0.5565521530460769


## t10a. Равномерный закон: Пирсон

In [ ]:
# t10a
# Для дискретного равномерного закона на {0,1,...,9} основной критерий согласия - критерий Пирсона.


m10 = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7], dtype=float)
n10 = m10.sum()

exp10a = np.full(10, n10 / 10)
delta10a_p = ((m10 - exp10a) ** 2 / exp10a).sum()
df10a_p = 10 - 1
pval10a_p = 1 - chi2.cdf(delta10a_p, df=df10a_p)

print("t10a")
print("n =", n10)
print("expected =", exp10a)
print("min expected =", exp10a.min())
print("Pearson delta =", delta10a_p)
print("Pearson df =", df10a_p)
print("Pearson p-value =", pval10a_p)

t10a
n = 100.0
expected = [10. 10. 10. 10. 10. 10. 10. 10. 10. 10.]
min expected = 10.0
Pearson delta = 16.4
Pearson df = 9
Pearson p-value = 0.058984030544419475


## t10b. Нормальный закон: ОМПГ, Пирсон, параметрический bootstrap для Колмогорова

In [6]:
# t10b
# Полная группа событий: A0 = (-inf, 1), A1 = [1, 2), ..., A8 = [8, 9), A9 = [9, +inf)
# Параметры нормального распределения ищутся только через ОМПГ: L(theta1, theta2) = prod p_i(theta1, theta2)^m_i

m10b = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7], dtype=float)
n10b = int(m10b.sum())

edges10b = np.array([
    -np.inf, 1, 2, 3, 4, 5, 6, 7, 8, 9, np.inf
], dtype=float)


def norm_group_probs(a, sigma):
    if sigma <= 0:
        return None
    cdf_vals = norm.cdf(edges10b, loc=a, scale=sigma)
    p = np.diff(cdf_vals)
    return p


def neg_loglik_t10b(params, counts):
    a, log_sigma = params
    sigma = math.exp(log_sigma)
    p = norm_group_probs(a, sigma)
    if p is None or np.any(p <= 0):
        return np.inf
    return -(counts * np.log(p)).sum()


# Начальная точка нужна только для численного поиска максимума
x_start = np.arange(10, dtype=float)
a0 = (x_start * m10b).sum() / n10b
s0 = math.sqrt((((x_start - a0) ** 2) * m10b).sum() / n10b)

res10b = minimize(
    neg_loglik_t10b,
    x0=np.array([a0, math.log(s0)]),
    args=(m10b,),
    method="Nelder-Mead",
    options={"maxiter": 20000, "xatol": 1e-12, "fatol": 1e-12}
)

a_hat_10b = float(res10b.x[0])
sigma_hat_10b = float(math.exp(res10b.x[1]))

p10b = norm_group_probs(a_hat_10b, sigma_hat_10b)
exp10b = n10b * p10b

delta10b_p = ((m10b - exp10b) ** 2 / exp10b).sum()
df10b_p = 10 - 1 - 2
pval10b_p = 1 - chi2.cdf(delta10b_p, df=df10b_p)

print("t10b Pearson")
print("a_hat (ОМПГ) =", a_hat_10b)
print("sigma_hat (ОМПГ) =", sigma_hat_10b)
print("probabilities =", p10b)
print("expected =", exp10b)
print("min expected =", exp10b.min())
print("delta =", delta10b_p)
print("df =", df10b_p)
print("p-value =", pval10b_p)

t10b Pearson
a_hat (ОМПГ) = 5.28967682492434
sigma_hat (ОМПГ) = 2.679519647435194
probabilities = [0.054698 0.05508  0.086634 0.118737 0.141807 0.147576 0.133828 0.105751
 0.072817 0.083073]
expected = [ 5.469814  5.507953  8.663352 11.873728 14.180665 14.757616 13.382779
 10.575136  7.281702  8.307255]
min expected = 5.469813677201482
delta = 9.802554904245365
df = 7
p-value = 0.20004135372128395


In [7]:
# t10b Kolmogorov + parametric bootstrap
# В каждой bootstrap-выборке параметры переоцениваются тем же методом, что и на исходной выборке, т.е. через ОМПГ по той же полной группе событий.

x_obs = np.repeat(np.arange(10), m10b.astype(int))


def fit_grouped_normal_ompg_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    x = np.arange(10, dtype=float)

    a_start = (x * counts).sum() / counts.sum()
    s_start = math.sqrt((((x - a_start) ** 2) * counts).sum() / counts.sum())

    res = minimize(
        neg_loglik_t10b,
        x0=np.array([a_start, math.log(s_start)]),
        args=(counts,),
        method="Nelder-Mead",
        options={"maxiter": 20000, "xatol": 1e-10, "fatol": 1e-10}
    )
    a_hat = float(res.x[0])
    sigma_hat = float(math.exp(res.x[1]))
    return a_hat, sigma_hat


def kolmogorov_statistic(sample, a, sigma):
    sample = np.sort(np.asarray(sample, dtype=float))
    n = len(sample)

    F = norm.cdf(sample, loc=a, scale=sigma)
    Fn_right = np.arange(1, n + 1) / n
    Fn_left = np.arange(0, n) / n

    D = max(
        np.max(np.abs(Fn_right - F)),
        np.max(np.abs(Fn_left - F))
    )
    return math.sqrt(n) * D


delta_obs_10b_k = kolmogorov_statistic(x_obs, a_hat_10b, sigma_hat_10b)

rng = np.random.default_rng(42)
B = 5000
delta_star = np.empty(B, dtype=float)

for b in range(B):
    sample_star = rng.normal(loc=a_hat_10b, scale=sigma_hat_10b, size=n10b)

    counts_star, _ = np.histogram(sample_star, bins=edges10b)

    a_star, sigma_star = fit_grouped_normal_ompg_from_counts(counts_star)

    x_star_grouped = np.repeat(np.arange(10), counts_star)

    delta_star[b] = kolmogorov_statistic(x_star_grouped, a_star, sigma_star)

m_exceed = np.sum(delta_star >= delta_obs_10b_k)
pval10b_k = m_exceed / B

print("t10b Kolmogorov + parametric bootstrap")
print("observed statistic =", delta_obs_10b_k)
print("bootstrap repetitions =", B)
print("m =", m_exceed, "(number of bootstrap statistics >= observed statistic)")
print("bootstrap p-value =", pval10b_k)

t10b Kolmogorov + parametric bootstrap
observed statistic = 1.7304488745586593
bootstrap repetitions = 5000
m = 1780 (number of bootstrap statistics >= observed statistic)
bootstrap p-value = 0.356


**В т10 нет веских оснований отвергнуть гипотезу о нормальном распределении**